# Breast Cancer Image Segmentation with Attention U-Net

This notebook trains an **Attention U-Net** to segment tumor regions in breast ultrasound images from the **BUSI** (Breast Ultrasound Images) dataset. It is configured to run on **Google Colab** with the dataset stored on Google Drive.

**Pipeline overview:**
1. Mount Google Drive and configure paths
2. Load and preprocess images & masks (resize to 256×256, normalize to [0, 1])
3. Merge multi-mask samples into a single mask per image
4. Build the Attention U-Net (5-level encoder/decoder + 4 attention gates)
5. Train with `binary_crossentropy` loss
6. Evaluate with confusion matrix, pixel-level metrics, and per-image Dice/IoU

**Dataset:** The BUSI dataset contains breast ultrasound images organized into three folders: `benign`, `malignant`, and `normal`. Each image has a corresponding binary mask (`*_mask.png`) marking the tumor region. Some samples have additional masks (`*_mask_1.png`) for multi-region lesions; we merge these into a single mask.


---
## 1. Colab Setup

Run the cells below in order: mount Drive, configure paths, and check GPU availability.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# === Path configuration ===
# Edit DRIVE_FOLDER if your dataset is stored at a different location.
import os

DRIVE_FOLDER = '/content/drive/MyDrive/breast cancer- segment'
DATASET_NAME = 'Dataset_BUSI_with_GT'

# Root path used by all data-loading cells
root_path = os.path.join(DRIVE_FOLDER, DATASET_NAME) + '/'

# Where to save the best model weights (kept on Drive so they survive Colab disconnects)
MODEL_SAVE_PATH = os.path.join(DRIVE_FOLDER, 'AttentionCustomUNet.h5')

# Sanity check
assert os.path.exists(root_path), f'Dataset not found at: {root_path}'
print('Dataset path :', root_path)
print('Subfolders   :', sorted(os.listdir(root_path)))
print('Model output :', MODEL_SAVE_PATH)


In [ ]:
# Check GPU availability
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {gpus}')
else:
    print('No GPU detected. Training will be slow.')
    print('Go to: Runtime -> Change runtime type -> select GPU (T4 / A100).')


---
## 2. Imports

Install `tf_explain` for GradCAM visualization (optional). The import is wrapped in `try/except` so the notebook still runs if the package is incompatible with the current TensorFlow version.


In [ ]:
from IPython.display import clear_output
!pip install tf_explain -q
clear_output()


In [ ]:
# Standard
import os
import numpy as np
import pandas as pd
from glob import glob

# TensorFlow / Keras
import tensorflow as tf
import tensorflow.image as tfi
import keras
from keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Evaluation
from sklearn.metrics import confusion_matrix

# Model layers
from keras.models import Model
from keras.layers import (
    Layer, Conv2D, Dropout, UpSampling2D,
    concatenate, Add, Multiply, Input,
    MaxPool2D, BatchNormalization
)

# Callbacks & metrics
from keras.callbacks import Callback, EarlyStopping, ModelCheckpoint
from keras.metrics import MeanIoU

# Optional: GradCAM via tf_explain
try:
    from tf_explain.core.grad_cam import GradCAM
    HAS_TF_EXPLAIN = True
    print('tf_explain available -- GradCAM will be shown during training.')
except Exception as e:
    HAS_TF_EXPLAIN = False
    print(f'tf_explain unavailable ({type(e).__name__}) -- GradCAM will be skipped.')


---
## 3. Helper Functions

`load_image` reads a single image and resizes it to `(SIZE, SIZE)`, normalizing pixel values to `[0, 1]`.

`load_images` batches this over a list of paths, returning either RGB images `(N, SIZE, SIZE, 3)` or single-channel masks `(N, SIZE, SIZE, 1)`.


In [ ]:
def load_image(image, SIZE):
    """Load and resize a single image to (SIZE, SIZE), normalized to [0, 1]."""
    return np.round(tfi.resize(img_to_array(load_img(image)) / 255., (SIZE, SIZE)), 4)


def load_images(image_paths, SIZE, mask=False, trim=None):
    """Load a batch of images or masks into a single numpy array."""
    if trim is not None:
        image_paths = image_paths[:trim]

    if mask:
        images = np.zeros(shape=(len(image_paths), SIZE, SIZE, 1))
    else:
        images = np.zeros(shape=(len(image_paths), SIZE, SIZE, 3))

    for i, image in enumerate(image_paths):
        img = load_image(image, SIZE)
        if mask:
            images[i] = img[:, :, :1]
        else:
            images[i] = img

    return images


In [ ]:
def show_image(image, title=None, cmap=None, alpha=1):
    """Display a single image without axes."""
    plt.imshow(image, cmap=cmap, alpha=alpha)
    if title is not None:
        plt.title(title)
    plt.axis('off')


def show_mask(image, mask, cmap=None, alpha=0.4):
    """Overlay a mask on top of an image."""
    plt.imshow(image)
    plt.imshow(tf.squeeze(mask), cmap=cmap, alpha=alpha)
    plt.axis('off')


---
## 4. Data Preparation

We collect all `(image, mask)` path pairs from the three class folders (`benign`, `malignant`, `normal`). Multi-mask samples (those with `*_mask_1.png`) are tracked separately and merged later.


In [ ]:
# Image resolution used throughout
SIZE = 256


In [ ]:
# List class folders
classes = sorted(os.listdir(root_path))
print('Classes:', classes)


In [ ]:
# Collect mask file paths
# - single_mask_paths: every '*_mask.png' (every sample has one)
# - double_mask_paths: '*_mask_1.png' (extra mask for multi-region lesions)
single_mask_paths = sorted([sorted(glob(root_path + name + "/*mask.png"))   for name in classes])
double_mask_paths = sorted([sorted(glob(root_path + name + "/*mask_1.png")) for name in classes])


In [ ]:
# Build (image_path, mask_path) lists from the single-mask set
image_paths = []
mask_paths  = []
for class_path in single_mask_paths:
    for path in class_path:
        img_path = path.replace('_mask', '')
        image_paths.append(img_path)
        mask_paths.append(path)

print(f'Total image-mask pairs: {len(image_paths)}')


In [ ]:
# Preview a sample: image alone, then image with mask overlay
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
show_image(load_image(image_paths[0], SIZE), title='Image')

plt.subplot(1, 2, 2)
show_mask(load_image(image_paths[0], SIZE),
          load_image(mask_paths[0], SIZE)[:, :, 0],
          alpha=0.6)
plt.title('Image + Mask Overlay')

plt.tight_layout()
plt.show()


---
## 5. Handling Multi-Mask Samples

A small number of samples have **two separate masks** for the same lesion (e.g. `benign (100)_mask.png` and `benign (100)_mask_1.png`). Since both masks belong to the **same class** (tumor), we merge them with element-wise addition. Only the first channel of the merged result is kept since masks are inherently single-channel.

Below we visualize this for `benign (100)`.


In [ ]:
# Example: a sample with two masks
example_base = root_path + 'benign/benign (100)'

original = load_image(example_base + '.png',          SIZE)
mask_a   = load_image(example_base + '_mask.png',     SIZE)
mask_b   = load_image(example_base + '_mask_1.png',   SIZE)

# Merge: simple addition (clipped implicitly by overlay)
merged = (mask_a + mask_b)[:, :, 0]

plt.figure(figsize=(16, 4))
plt.subplot(1, 4, 1); show_image(original,        title='Original Image')
plt.subplot(1, 4, 2); show_image(mask_a[:, :, 0], title='Mask 1',  cmap='gray')
plt.subplot(1, 4, 3); show_image(mask_b[:, :, 0], title='Mask 2',  cmap='gray')
plt.subplot(1, 4, 4); show_image(merged,          title='Merged Mask', cmap='gray')
plt.tight_layout()
plt.show()

# Overlay the merged mask on the original
plt.figure(figsize=(5, 5))
show_image(original, title='Original + Merged Mask')
plt.imshow(merged, cmap='gray', alpha=0.4)
plt.axis('off')
plt.show()


> Note: there are only ~16 samples with double masks. Since they are few, dropping or merging them does not significantly affect training. We use **only the single-mask path list** in the next step (multi-mask samples still get their primary `_mask.png` included).


---
## 6. Load All Images & Masks Into Memory

The full dataset fits comfortably in RAM at 256×256, so we load everything up front. This makes training faster (no disk I/O per batch) and lets us index into `images` / `masks` directly inside callbacks.


In [ ]:
images = load_images(image_paths, SIZE)
masks  = load_images(mask_paths,  SIZE, mask=True)

print(f'images shape: {images.shape}')
print(f'masks  shape: {masks.shape}')


In [ ]:
# Random sample grid: image with mask overlay
plt.figure(figsize=(13, 8))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    idx = np.random.randint(len(images))
    show_mask(images[idx], masks[idx], cmap='jet')
plt.suptitle('Random samples: image + mask overlay (cmap=jet)', y=1.02)
plt.tight_layout()
plt.show()


---
## 7. Model Components

The Attention U-Net consists of three custom layers:

- **EncoderBlock** — two `Conv2D + ReLU` layers with dropout, optionally followed by `MaxPool2D`. Returns both the pooled output (for the next encoder level) and the un-pooled output (skip connection).
- **DecoderBlock** — `UpSampling2D`, concatenation with a skip connection, then a non-pooling `EncoderBlock` for further refinement.
- **AttentionGate** — additive attention mechanism. Takes the deeper feature map `X` and a skip connection `skip_X`, learns a spatial attention map (sigmoid), and uses it to gate the skip features before they are passed to the decoder.

Each layer implements `get_config` so the model can be serialized/loaded.


In [ ]:
class EncoderBlock(Layer):
    """Two Conv-ReLU layers with dropout, optional max-pooling.

    Returns (pooled, pre_pool) when pooling=True (pre_pool is a skip connection),
    otherwise just the pre-pool feature map.
    """

    def __init__(self, filters, rate, pooling=True, **kwargs):
        super(EncoderBlock, self).__init__(**kwargs)
        self.filters = filters
        self.rate    = rate
        self.pooling = pooling

        self.c1   = Conv2D(filters, kernel_size=3, strides=1, padding='same',
                           activation='relu', kernel_initializer='he_normal')
        self.drop = Dropout(rate)
        self.c2   = Conv2D(filters, kernel_size=3, strides=1, padding='same',
                           activation='relu', kernel_initializer='he_normal')
        self.pool = MaxPool2D()

    def call(self, X):
        x = self.c1(X)
        x = self.drop(x)
        x = self.c2(x)
        if self.pooling:
            y = self.pool(x)
            return y, x
        else:
            return x

    def get_config(self):
        base_config = super().get_config()
        return {**base_config,
                'filters': self.filters,
                'rate':    self.rate,
                'pooling': self.pooling}


In [ ]:
class DecoderBlock(Layer):
    """UpSample -> concat with skip -> EncoderBlock(pooling=False)."""

    def __init__(self, filters, rate, **kwargs):
        super(DecoderBlock, self).__init__(**kwargs)
        self.filters = filters
        self.rate    = rate

        self.up  = UpSampling2D()
        self.net = EncoderBlock(filters, rate, pooling=False)

    def call(self, X):
        X, skip_X = X
        x  = self.up(X)
        c_ = concatenate([x, skip_X])
        x  = self.net(c_)
        return x

    def get_config(self):
        base_config = super().get_config()
        return {**base_config,
                'filters': self.filters,
                'rate':    self.rate}


In [ ]:
class AttentionGate(Layer):
    """Additive attention gate that filters skip-connection features."""

    def __init__(self, filters, bn, **kwargs):
        super(AttentionGate, self).__init__(**kwargs)
        self.filters = filters
        self.bn      = bn

        self.normal   = Conv2D(filters, kernel_size=3,            padding='same',
                               activation='relu',    kernel_initializer='he_normal')
        self.down     = Conv2D(filters, kernel_size=3, strides=2, padding='same',
                               activation='relu',    kernel_initializer='he_normal')
        self.learn    = Conv2D(1,       kernel_size=1,            padding='same',
                               activation='sigmoid')
        self.resample = UpSampling2D()
        self.BN       = BatchNormalization()

    def call(self, X):
        X, skip_X = X

        x    = self.normal(X)
        skip = self.down(skip_X)
        x    = Add()([x, skip])
        x    = self.learn(x)
        x    = self.resample(x)
        f    = Multiply()([x, skip_X])
        if self.bn:
            return self.BN(f)
        return f

    def get_config(self):
        base_config = super().get_config()
        return {**base_config,
                'filters': self.filters,
                'bn':      self.bn}


---
## 8. Training Callback

`ShowProgress` runs at the end of every epoch and visualizes (a) the ground-truth mask, (b) the predicted mask, and optionally (c) a GradCAM activation map for the `Attention4` layer. Useful for sanity-checking that the model is actually learning to localize tumors during training.


In [ ]:
class ShowProgress(Callback):
    """Visualize a random sample's prediction at the end of every epoch."""

    def on_epoch_end(self, epochs, logs=None):
        idx = np.random.randint(min(200, len(images)))
        image     = images[idx]
        mask      = masks[idx]
        pred_mask = self.model.predict(image[np.newaxis, ...], verbose=0)

        n_plots = 3 if HAS_TF_EXPLAIN else 2
        plt.figure(figsize=(10, 5))

        plt.subplot(1, n_plots, 1)
        plt.title('Ground-Truth Mask')
        show_mask(image, mask, cmap='copper')

        plt.subplot(1, n_plots, 2)
        plt.title('Predicted Mask')
        show_mask(image, pred_mask, cmap='copper')

        if HAS_TF_EXPLAIN:
            try:
                exp = GradCAM()
                cam = exp.explain(
                    validation_data=(image[np.newaxis, ...], mask),
                    class_index=1,
                    layer_name='Attention4',
                    model=self.model,
                )
                plt.subplot(1, 3, 3)
                show_image(cam, title='GradCAM (Attention4)')
            except Exception as e:
                print(f'(GradCAM skipped: {e})')

        plt.tight_layout()
        plt.show()


---
## 9. Build the Attention U-Net

Architecture summary:

| Stage    | Filters | Pooling | Dropout |
|----------|---------|---------|---------|
| Encoder1 | 32      | yes     | 0.1     |
| Encoder2 | 64      | yes     | 0.1     |
| Encoder3 | 128     | yes     | 0.2     |
| Encoder4 | 256     | yes     | 0.2     |
| Encoding | 512     | no      | 0.3     |
| Decoder1 | 256     | -       | 0.2     |
| Decoder2 | 128     | -       | 0.2     |
| Decoder3 | 64      | -       | 0.1     |
| Decoder4 | 32      | -       | 0.1     |

- **Loss:** `binary_crossentropy` (per-pixel)
- **Optimizer:** `adam`
- **Metrics:** `accuracy`, `MeanIoU(num_classes=2)`
- **Output:** `1×1 Conv2D` with `sigmoid` -> per-pixel tumor probability


In [ ]:
# === Inputs ===
input_layer = Input(shape=images.shape[-3:])

# === Encoder ===
p1, c1 = EncoderBlock(32,  0.1, name='Encoder1')(input_layer)
p2, c2 = EncoderBlock(64,  0.1, name='Encoder2')(p1)
p3, c3 = EncoderBlock(128, 0.2, name='Encoder3')(p2)
p4, c4 = EncoderBlock(256, 0.2, name='Encoder4')(p3)

# === Bottleneck ===
encoding = EncoderBlock(512, 0.3, pooling=False, name='Encoding')(p4)

# === Attention + Decoder ===
a1 = AttentionGate(256, bn=True, name='Attention1')([encoding, c4])
d1 = DecoderBlock(256, 0.2,      name='Decoder1')([encoding, a1])

a2 = AttentionGate(128, bn=True, name='Attention2')([d1, c3])
d2 = DecoderBlock(128, 0.2,      name='Decoder2')([d1, a2])

a3 = AttentionGate(64, bn=True,  name='Attention3')([d2, c2])
d3 = DecoderBlock(64, 0.1,       name='Decoder3')([d2, a3])

a4 = AttentionGate(32, bn=True,  name='Attention4')([d3, c1])
d4 = DecoderBlock(32, 0.1,       name='Decoder4')([d3, a4])

# === Output ===
output_layer = Conv2D(1, kernel_size=1, activation='sigmoid', padding='same')(d4)

# === Compile ===
model = Model(inputs=[input_layer], outputs=[output_layer])
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', MeanIoU(num_classes=2, name='IoU')],
)

# === Callbacks ===
cb = [
    # EarlyStopping(patience=3, restore_best_weights=True),
    # For segmentation we trust visual inspection more than metrics,
    # so we just save the best weights and watch progress visually.
    ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True),
    ShowProgress(),
]

model.summary()


---
## 10. Training

20 epochs is usually enough for the model to start producing usable masks (~12 epochs in). For better quality go to 25-30+ epochs.


In [ ]:
# Training config
BATCH_SIZE = 8
SPE        = len(images) // BATCH_SIZE   # steps per epoch

results = model.fit(
    images, masks,
    validation_split=0.2,
    epochs=20,
    steps_per_epoch=SPE,
    batch_size=BATCH_SIZE,
    callbacks=cb,
)


---
## 11. Training Curves

Plot loss, accuracy, and IoU for both training and validation across epochs.


In [ ]:
# Pull metric history (using explicit keys is safer than .values() unpacking)
loss         = results.history['loss']
val_loss     = results.history['val_loss']
accuracy     = results.history['accuracy']
val_accuracy = results.history['val_accuracy']
iou          = results.history['IoU']
val_iou      = results.history['val_IoU']

plt.figure(figsize=(20, 5))

plt.subplot(1, 3, 1)
plt.title('Loss')
plt.plot(loss,     label='Training')
plt.plot(val_loss, label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid()

plt.subplot(1, 3, 2)
plt.title('Accuracy')
plt.plot(accuracy,     label='Training')
plt.plot(val_accuracy, label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid()

plt.subplot(1, 3, 3)
plt.title('Mean IoU')
plt.plot(iou,     label='Training')
plt.plot(val_iou, label='Validation')
plt.xlabel('Epoch'); plt.ylabel('IoU')
plt.legend(); plt.grid()

plt.tight_layout()
plt.show()


---
## 12. Evaluation

We evaluate the model on the validation split (the **last 20%** of `images` / `masks`, which is what `validation_split=0.2` uses internally). The reported quantities are:

- **Pixel-level confusion matrix** — TP / FP / TN / FN summed across all pixels of all validation images
- **Pixel-level metrics** — Accuracy (primary), Precision, Recall (Sensitivity), Specificity, Dice (F1), IoU
- **Per-image distributions** — histograms of Dice and IoU computed image-by-image


In [ ]:
# === Validation set evaluation: confusion matrix + metrics + per-image distributions ===

# 1. Reconstruct the validation split (Keras validation_split=0.2 takes the last 20%)
val_split = 0.2
n_val     = int(len(images) * val_split)
val_images = images[-n_val:]
val_masks  = masks[-n_val:]
print(f'Validation set size: {len(val_images)}')

# 2. Predict on the validation set and threshold at 0.5
val_preds        = model.predict(val_images, batch_size=8, verbose=1)
val_preds_binary = (val_preds > 0.5).astype(np.uint8)
val_masks_binary = (val_masks > 0.5).astype(np.uint8)

# 3. Pixel-level confusion matrix and derived metrics
y_true = val_masks_binary.flatten()
y_pred = val_preds_binary.flatten()
cm     = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy    = (tp + tn) / (tp + tn + fp + fn)
precision   = tp / (tp + fp) if (tp + fp) > 0 else 0
recall      = tp / (tp + fn) if (tp + fn) > 0 else 0   # = Sensitivity
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
dice        = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
iou_pixel   = tp / (tp + fp + fn)         if (tp + fp + fn)     > 0 else 0

print('\n=== Validation Metrics (pixel-level) ===')
print(f'>>> Accuracy   : {accuracy:.4f}   <<<   (primary metric)')
print(f'    Precision  : {precision:.4f}')
print(f'    Recall     : {recall:.4f}   (Sensitivity)')
print(f'    Specificity: {specificity:.4f}')
print(f'    Dice (F1)  : {dice:.4f}')
print(f'    IoU        : {iou_pixel:.4f}')

# 4. Plot the confusion matrix (raw counts and row-normalized)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Background', 'Tumor'],
            yticklabels=['Background', 'Tumor'], ax=axes[0])
axes[0].set_title('Confusion Matrix (Pixel Counts)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.4f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=['Background', 'Tumor'],
            yticklabels=['Background', 'Tumor'], ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized by True Class)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

# 5. Per-image Dice / IoU distributions
def per_image_metrics(true_masks, pred_masks, eps=1e-7):
    """Compute Dice and IoU for each image individually.

    Edge cases:
      - both ground truth and prediction empty -> score = 1.0
      - exactly one of them empty              -> score = 0.0
    """
    dices, ious = [], []
    for t, p in zip(true_masks, pred_masks):
        t = t.flatten().astype(np.uint8)
        p = p.flatten().astype(np.uint8)
        inter = (t & p).sum()
        union = (t | p).sum()
        if t.sum() == 0 and p.sum() == 0:
            d, j = 1.0, 1.0
        elif t.sum() == 0 or p.sum() == 0:
            d, j = 0.0, 0.0
        else:
            d = 2 * inter / (t.sum() + p.sum() + eps)
            j = inter     / (union              + eps)
        dices.append(d)
        ious.append(j)
    return np.array(dices), np.array(ious)


dices, ious = per_image_metrics(val_masks_binary, val_preds_binary)

print(f'\n=== Per-Image Metrics (n={len(dices)}) ===')
print(f'Dice  - mean: {dices.mean():.4f}, median: {np.median(dices):.4f}, std: {dices.std():.4f}')
print(f'IoU   - mean: {ious.mean():.4f},  median: {np.median(ious):.4f},  std: {ious.std():.4f}')
print(f'Fraction with Dice > 0.7: {(dices > 0.7).mean() * 100:.1f}%')
print(f'Fraction with Dice > 0.5: {(dices > 0.5).mean() * 100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(dices, bins=30, color='steelblue', edgecolor='black')
axes[0].axvline(dices.mean(), color='red', linestyle='--', label=f'Mean = {dices.mean():.3f}')
axes[0].set_title('Per-Image Dice Distribution')
axes[0].set_xlabel('Dice'); axes[0].set_ylabel('Count'); axes[0].legend()

axes[1].hist(ious, bins=30, color='coral', edgecolor='black')
axes[1].axvline(ious.mean(), color='red', linestyle='--', label=f'Mean = {ious.mean():.3f}')
axes[1].set_title('Per-Image IoU Distribution')
axes[1].set_xlabel('IoU'); axes[1].set_ylabel('Count'); axes[1].legend()

plt.tight_layout()
plt.show()


---
## 13. Sample Predictions

Visualize 5 random samples in three columns: ground-truth mask, raw predicted mask (sigmoid output), and binarized predicted mask (threshold = 0.5).


In [ ]:
plt.figure(figsize=(20, 25))
n = 0
for i in range(1, (5 * 3) + 1):
    plt.subplot(5, 3, i)

    if n == 0:
        idx       = np.random.randint(len(images))
        image     = images[idx]
        mask      = masks[idx]
        pred_mask = model.predict(image[np.newaxis, ...], verbose=0)

        plt.title('Ground-Truth Mask')
        show_mask(image, mask)
        n += 1

    elif n == 1:
        plt.title('Predicted Mask (raw)')
        show_mask(image, pred_mask)
        n += 1

    elif n == 2:
        pred_mask_bin = (pred_mask > 0.5).astype('float')
        plt.title('Predicted Mask (thresholded @ 0.5)')
        show_mask(image, pred_mask_bin)
        n = 0

plt.tight_layout()
plt.show()


---
### End of notebook

The best model weights are saved at `MODEL_SAVE_PATH` (on Drive). To reload:

```python
from keras.models import load_model
model = load_model(MODEL_SAVE_PATH, custom_objects={
    'EncoderBlock':   EncoderBlock,
    'DecoderBlock':   DecoderBlock,
    'AttentionGate':  AttentionGate,
})
```
